# 02_ex_agreement_topic — exploration and proposal

Analyst/data scientist notebook for source exploration, exploratory transforms, and proposal evidence.

> AI output in this notebook is advisory and requires human validation.
> Governance approval belongs to `01_data_sharing_agreement`; production enforcement belongs to `03_pc`.


## 01 Introduction and scope

Document:
- `agreement_id`
- `topic`
- source being explored
- question being answered
- approved usage context from the data sharing agreement


In [ ]:
agreement_id = "REPLACE_AGREEMENT_ID"
topic = "REPLACE_TOPIC"
table_name = "REPLACE_TABLE_NAME_OR_ALIAS"


## 02 Agreement context

Capture approved context from `01_data_sharing_agreement_<agreement>`:
- approved usage scope
- stakeholder owner/steward
- known restrictions relevant to this exploration


## 03 Configuration and setup


In [ ]:
%run 00_env_config


## 04 Metadata-first inspection

Before profiling the source again, check what is already known. Existing metadata may already contain profile evidence, approved DQ rules, sensitivity labels, business context, and prior proposal evidence.


In [ ]:
from fabricops_kit import get_path, lakehouse_table_read

metadata_lakehouse = get_path(ENV_NAME, "metadata", config=FABRIC_CONFIG)

# Optional existing metadata reads. Keep table names explicit and easy to edit.
# Uncomment the tables your project already maintains.
# existing_profile_df = lakehouse_table_read(metadata_lakehouse, "profile_rows")
# existing_dq_df = lakehouse_table_read(metadata_lakehouse, "dq_approved")
# existing_governance_df = lakehouse_table_read(metadata_lakehouse, "governance_approved")
# existing_business_context_df = lakehouse_table_read(metadata_lakehouse, "business_context_approved")
# existing_proposal_df = lakehouse_table_read(metadata_lakehouse, "proposal_evidence")

# Optional table-scoped inspection example:
# existing_profile_df.filter(existing_profile_df.TABLE_NAME == table_name).show(truncate=False)


## 05 Reuse/refresh decision

Decision: reuse existing metadata, refresh the profile, or create new proposal evidence.

Checklist:
- Existing profile is available and current enough
- Existing approved DQ rules already cover the known risks
- Existing governance labels are sufficient
- New source behavior or business question requires fresh profiling
- New proposal evidence is required


## 06 Data ingestion for exploration

Choose one source loading pattern only when a fresh read is needed.


In [ ]:
from fabricops_kit import (
    get_path,
    lakehouse_table_read,
    lakehouse_csv_read,
    lakehouse_parquet_read_as_spark,
    warehouse_read,
    profile_dataframe,
)

# Choose one source loading pattern only when a fresh read is needed.

# Option A: Lakehouse Delta table
source_df = lakehouse_table_read(
    get_path(ENV_NAME, "source", config=FABRIC_CONFIG),
    table_name,
)

# Option B: Warehouse table
# source_df = warehouse_read(
#     env=ENV_NAME,
#     target="warehouse",
#     schema="dbo",
#     table=table_name,
#     config=FABRIC_CONFIG,
# )

# Option C: Lakehouse CSV file
# source_df = lakehouse_csv_read(
#     get_path(ENV_NAME, "source", config=FABRIC_CONFIG),
#     "Files/REPLACE_PATH.csv",
# )

# Option D: Lakehouse Parquet file
# source_df = lakehouse_parquet_read_as_spark(
#     get_path(ENV_NAME, "source", config=FABRIC_CONFIG),
#     "Files/REPLACE_PATH.parquet",
# )

source_df.printSchema()
source_df.limit(20).show(truncate=False)


## 07 Source profiling (optional refresh)

Use profiling when existing metadata is missing, stale, or insufficient for the current question.


In [ ]:
profile_df = profile_dataframe(source_df, table_name=table_name)
profile_df.show(50, truncate=False)

# profile_df.write.mode("append").saveAsTable("REPLACE_PROFILE_OUTPUT_TABLE")


## 08 Exploratory analysis

Inspect distributions, nulls, duplicates, sample records, and capture analyst observations.


In [ ]:
# Example exploration snippets
source_df.selectExpr("count(*) as total_rows").show()
source_df.limit(20).show(truncate=False)

# TODO: add focused analyst observations for this topic.


## 09 AI-assisted DQ rule review (optional)

Optional analyst/engineer-owned DQ flow: load existing approved rules, draft AI candidates when needed, review in the existing DQ widget, persist approved rules, review deactivations, and persist deactivation metadata.


In [ ]:
from fabricops_kit import (
    draft_dq_rules,
    run_dq_rule_review_widget,
    get_dq_rule_review_results,
    write_dq_rules,
    load_approved_dq_rules,
    review_dq_rule_deactivations,
    build_dq_rule_deactivation_metadata_df,
)

# 1) Optional: load existing approved/active DQ rules for this table from metadata
# existing_dq_df = lakehouse_table_read(metadata_lakehouse, "dq_rules")
# approved_active_rules = load_approved_dq_rules(existing_dq_df, table_name=table_name)

# 2) Optional: generate AI candidate rules only when needed
# profile_for_ai_df = existing_profile_df  # if current enough
# profile_for_ai_df = profile_df           # if refreshed
# candidate_rules_df = draft_dq_rules(profile_df=profile_for_ai_df, table_name=table_name)

# 3) Optional: analyst/engineer review/edit/approve via existing DQ widget
# run_dq_rule_review_widget(candidate_rules_df, table_name=table_name)
# review_results = get_dq_rule_review_results(table_name=table_name)

# 4) Optional: persist approved rules to append-only DQ metadata
# approved_rules = review_results.get("approved_rules", [])
# if approved_rules:
#     write_dq_rules(approved_rules, table_name=table_name, metadata_path=metadata_lakehouse)

# 5) Optional: review existing active rules for deactivation with explicit reasons
# deactivation_reviews = review_dq_rule_deactivations(approved_active_rules, table_name=table_name)

# 6) Optional: build and persist deactivation metadata rows
# deactivation_df = build_dq_rule_deactivation_metadata_df(
#     deactivation_reviews,
#     table_name=table_name,
# )
# deactivation_df.write.mode("append").saveAsTable("REPLACE_DQ_RULES_METADATA_TABLE")

# 7) `03_pc` later loads active approved rules only and enforces deterministically.


## 10 Findings and proposed metadata updates


In [ ]:
proposal_rows = [
    {
        "agreement_id": agreement_id,
        "topic": topic,
        "table_name": table_name,
        "proposal_type": "dq_candidate",
        "rationale": "REPLACE",
        "source_evidence": "existing metadata / fresh profile / analyst observation",
    },
    {
        "agreement_id": agreement_id,
        "topic": topic,
        "table_name": table_name,
        "proposal_type": "classification_candidate",
        "rationale": "REPLACE",
        "source_evidence": "existing metadata / fresh profile / analyst observation",
    },
]

proposal_df = spark.createDataFrame(proposal_rows)
proposal_df.show(truncate=False)

# proposal_df.write.mode("append").saveAsTable("REPLACE_PROPOSAL_EVIDENCE_TABLE")


## 11 Handoff to governance and pipeline contract

- Governance updates move to `01_data_sharing_agreement_<agreement>` for approval.
- Approved metadata/rules/classifications are consumed by `03_pc_<agreement>_<pipeline>`.
- Production transforms and enforcement happen in `03_pc`.


## 12 Frozen diagnostics / appendix

Keep one-time diagnostics and deep debug outputs here. Avoid turning exploratory cells into scheduled production behavior in `02_ex`.
